Lazada Scraper

In [ ]:
import sys
print(sys.executable)  # Should show path inside your /venv/ folder

In [ ]:
import time
import random
import numpy as np
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
PRODUCTS = {
    "Baju": [
        "https://www.lazada.co.id/products/pdp-i8204206549-s14611958962.html",
        "https://www.lazada.co.id/products/pdp-i8334366121-s14753990628.html",
        "https://www.lazada.co.id/products/pdp-i8065798509-s14724984412.html",
        "https://www.lazada.co.id/products/pdp-i7261980969-s14301254145.html", 
        "https://www.lazada.co.id/products/pdp-i8385068759-s116256725430.html",
        
    ],
    "Celana": [
        "https://www.lazada.co.id/products/pdp-i8329538333-s14748450790.html",
        "https://www.lazada.co.id/products/pdp-i8029448440-s14453508816.html",
        "https://www.lazada.co.id/products/pdp-i6109860480-s14747960455.html",
        "https://www.lazada.co.id/products/pdp-i8179442977-s14583920813.html",
        "https://www.lazada.co.id/products/pdp-i5719130159-s11170380904.html",
    ],
    "Kemeja": [
        "https://www.lazada.co.id/products/pdp-i7915018755-s14331480205.html",
        "https://www.lazada.co.id/products/pdp-i7453222631-s14615702214.html",
        "https://www.lazada.co.id/products/pdp-i7766026554-s14251158496.html",
        "https://www.lazada.co.id/products/pdp-i7894484306-s14309502832.html", 
        "https://www.lazada.co.id/products/pdp-i1209246242-s14683940315.html",
    ],
    "Dress": [
        "https://www.lazada.co.id/products/pdp-i8187198478-s116252884387.html",
        "https://www.lazada.co.id/products/pdp-i7288270589-s13782666588.html",
        "https://www.lazada.co.id/products/pdp-i8188508379-s14594458062.html",
        "https://www.lazada.co.id/products/pdp-i7807354672-s14260954831.html",
        "https://www.lazada.co.id/products/pdp-i8111146452-s14502426253.html",
    ],
}

In [ ]:
def create_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    return driver

In [ ]:
def scrape_product(driver, url, category, max_reviews=300):
    driver.get(url)
    time.sleep(random.uniform(4, 7))

    # slow scroll (anti-bot behavior)
    for _ in range(3):
        driver.execute_script("window.scrollBy(0, 600);")
        time.sleep(random.uniform(1, 2))

    reviews = []
    seen = set()
    wait = WebDriverWait(driver, 15)

    try:
        wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "div.pdp-mod-review-main")
            )
        )

        while len(reviews) < max_reviews:

            items = driver.find_elements(
                By.CSS_SELECTOR, "div.pdp-mod-review-main div.item"
            )

            if not items:
                break

            last_first = items[0].text

            for item in items:
                if len(reviews) >= max_reviews:
                    break

                # ======================
                # REVIEW TEXT
                # ======================
                try:
                    text = item.find_element(
                        By.CSS_SELECTOR,
                        "div.item-content-main-content-reviews span"
                    ).text.strip()

                    if text == "":
                        text = None
                except:
                    text = None

                # ======================
                # RATING (ALWAYS CAPTURE)
                # ======================
                rating = 0
                try:
                    stars = item.find_elements(
                        By.CSS_SELECTOR,
                        ".i-rate-star-item svg"
                    )

                    for s in stars:
                        paths = s.find_elements(By.TAG_NAME, "path")
                        if len(paths) > 1:
                            style = paths[1].get_attribute("style") or ""
                            if "255, 200, 60" in style:
                                rating += 1
                except:
                    rating = 0

                # ======================
                # DEDUP
                # ======================
                key = (text, rating)

                if key not in seen:
                    seen.add(key)

                    reviews.append({
                        "category": category,
                        "product_url": url,
                        "review": text if text is not None else np.nan,
                        "rating": rating
                    })

            print(f"📦 {url} → collected: {len(reviews)}/{max_reviews}")

            # ======================
            # PAGINATION
            # ======================
            clicked = False

            # 1. LOAD MORE
            try:
                load_more = driver.find_element(
                    By.XPATH,
                    "//button[contains(., 'Lihat Lebih Banyak')]"
                )
                driver.execute_script("arguments[0].click();", load_more)
                time.sleep(3)
                clicked = True
            except:
                pass

            # 2. NEXT PAGE
            if not clicked:
                try:
                    next_btn = driver.find_element(
                        By.XPATH,
                        "//li[contains(@class,'next')]//button"
                    )

                    if "disabled" not in next_btn.get_attribute("class"):
                        driver.execute_script(
                            "arguments[0].click();", next_btn
                        )
                        time.sleep(4)
                        clicked = True
                    else:
                        break
                except:
                    break

            if not clicked:
                break

            # ======================
            # STOP IF NO NEW DATA
            # ======================
            new_items = driver.find_elements(
                By.CSS_SELECTOR,
                "div.pdp-mod-review-main div.item"
            )

            if new_items and new_items[0].text == last_first:
                break

    except Exception as e:
        print("❌ Error:", e)

    return reviews

In [ ]:
def run_pipeline():
    driver = create_driver()

    CATEGORY_LIMIT = 1250
    GLOBAL_LIMIT = 5000

    final_data = []
    category_count = {c: 0 for c in PRODUCTS}

    try:
        for category, urls in PRODUCTS.items():

            print(f"\n🚀 CATEGORY START: {category}")

            for url in urls:

                if len(final_data) >= GLOBAL_LIMIT:
                    print("✅ Global limit reached (5000)")
                    return final_data

                if category_count[category] >= CATEGORY_LIMIT:
                    print(f"✅ Category {category} done (1250)")
                    break

                remaining = CATEGORY_LIMIT - category_count[category]
                fetch_limit = min(250, remaining)

                print(f"⏳ Scraping {url}")

                data = scrape_product(driver, url, category, fetch_limit)

                final_data.extend(data)
                category_count[category] += len(data)

                print(f"📦 {category}: {category_count[category]} collected")

                time.sleep(random.uniform(3, 6))

    finally:
        driver.quit()

    return final_data

In [ ]:
data = run_pipeline()

df = pd.DataFrame(data)
df.to_csv("lazada_5000_dataset.csv", index=False)

print("\n🎉 DONE")
print(df.head())